# P3 — smoke test cac candidate cai tien (15 epoch)

Nen chung: `uni_nokd` (w=(2,2,1,1)/6, KD=0). Moi candidate chi doi DUNG mot thanh phan.

| id | Doi gi | Uu tien |
|---|---|---|
| `base`   | khong doi (moc so sanh 15 epoch) | – |
| `g1`     | `gamma_img` 0.5 -> **1.0** (ve dung LoKU) | cao |
| `gate`   | Gate **co dinh + dung chung** thay vi ngau nhien moi batch | cao |
| `r16`    | `lora_r` 8->**16**, `lora_alpha` 16->**32** (giu alpha/r=2) | trung-cao |
| `more`   | them target LoRA: MLP cua SciBERT + 2 khoi conv cuoi | trung |
| `strong` | g1 + gate + r16 gop lai | chay sau |
| `strong_lr` | strong + `lr` 2e-4 -> **1e-3** | chi khi strong van yeu |

**Doi `ACCOUNT` o Cell 2 (1..5) roi Run All.** Moi account chay phan viec cua minh.

Tieu chi doc ket qua (Cell 5): `d_u_mean` co vuot 6.5-7.7 va **con tang o E15** khong?
Neu van phang -> candidate do that bai, khoi chay 30 epoch.


In [ ]:
# Cell 1: setup + CHOT CHAN code da push
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
_adv=open('training/adv_common.py').read()
assert 'ce_selector' in _adv and 'checkpoint_selection_' in _adv, \
    '❌ adv_common CHUA co hook CE-selector -> chay `git push` code MOI roi moi Save Version!'
assert 'OnlineCESelector' in open('training/ce_selector_pilot.py').read(), '❌ git push code moi truoc!'
assert os.path.exists('training/forgetmi_p3_cand.py'), '❌ chua push forgetmi_p3_cand.py!'
print('✅ Code CE-selector da co (hook + OnlineCESelector).')
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON + path discovery (mimic/iu) + ablation
import glob, os
DATASET    = 'mimic'   # 'mimic' | 'iu'
FORGET_PCT = 3         # 3 | 6 | 10  (iu: 3)
SEED       = 42

# --- DOI DUNG 1 DONG NAY tren moi account (1..5) ---
ACCOUNT = 1
EPOCHS  = 15          # smoke test; chi candidate thang moi chay lai 30
assert ACCOUNT in (1,2,3,4,5)

assert DATASET in ('mimic','iu') and FORGET_PCT in (3,6,10)
if DATASET=='iu': assert FORGET_PCT==3,'IU chi co 3%'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

tag=f'{DATASET}{FORGET_PCT}per'
OUT=f'/kaggle/working/p3imp_{tag}_s{SEED}'
RESULTS=f'/kaggle/working/results_p3imp_{tag}.csv'

if DATASET=='mimic':
    CONFIG='config_advanced_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gh[0]) if gh else BASE; HAS_GOLD=bool(gh)
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'
else:
    CONFIG='config_loku_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu + forget-mi-models-iu-re + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE; HAS_GOLD=bool(reb)
    def first_existing(root, rels):
        for r in rels:
            p=os.path.join(root,r)
            if os.path.exists(p): return p
        return None
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0]) if tsv else first_existing(DATA,['data/metadata','metadata'])
    IMG=first_existing(DATA,['data/img_data','img_data']) or (first_existing(RAD,['images/images_normalized','images']) if RAD else None) or RAD
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or glob.glob('/kaggle/input/**/iu-split.csv',recursive=True) or glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True) or glob.glob(f'/kaggle/input/**/forget_set_{FORGET_PCT}per_iu.csv',recursive=True)
    assert sp and fg,f'Khong thay iu-split / forget_set_iu (glob toan input)'
    SPLIT=sp[0]; FORGET=fg[0]
    if not TEXT or not IMG:
        print('⚠️ TEXT',TEXT,'IMG',IMG,'- liet ke input:')
        for r,d,f in os.walk(DATA):
            if r[len(DATA):].count(os.sep)<=2: print(' ',r,'->',[x for x in f][:4])

for n,p in {'BASE':BASE,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'
COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'results_csv_path':RESULTS,'ce_selector':1,'s4_delta':0.15,
        'use_noise':1}          # Forget-MI Eq.(1)-(2): reference = og(BAN NHIEU) - anh Gaussian + text perturb
print('DATASET',DATASET,'PCT',FORGET_PCT,'| config',CONFIG,'| tag',tag,'| GOLD',HAS_GOLD,'| ACCOUNT',ACCOUNT)
print('BASE',BASE); print('SPLIT',SPLIT); print('FORGET',FORGET)

# ---------------- DINH NGHIA CANDIDATE ----------------
# Moi candidate = (id, override them vao COMMON). Nen chung: scheme uni_nokd.
MLP_TXT = 'attention.output.dense|intermediate.dense|output.dense'
CANDS = {
 'base'     : {},
 'g1'       : {'loku_image_subtract_scale':1.0},
 'gate'     : {'gate_mode':'fixed_shared'},
 'r16'      : {'lora_r':16, 'lora_alpha':32},
 'more'     : {'lora_extra_target_modules':MLP_TXT, 'lora_image_last_k_blocks':2},
 'strong'   : {'loku_image_subtract_scale':1.0, 'gate_mode':'fixed_shared',
               'lora_r':16, 'lora_alpha':32},
 'strong_lr': {'loku_image_subtract_scale':1.0, 'gate_mode':'fixed_shared',
               'lora_r':16, 'lora_alpha':32, 'learning_rate':1e-3},
}
# Phan viec 5 account (~1.5h moi run)
PLAN = {1:['base','g1'], 2:['r16','more'], 3:['gate'], 4:['strong'], 5:['strong_lr']}
JOBS = PLAN[ACCOUNT]
print('ACCOUNT',ACCOUNT,'-> se chay:',JOBS,f'({EPOCHS} epoch moi run, ~{1.5*len(JOBS):.1f}h)')
for j in JOBS: print(f'   {j:10} {CANDS[j] if CANDS[j] else "(khong doi gi)"}')


In [ ]:
# Cell 3: chay lan luot cac candidate cua account nay
import os, subprocess, time
LOG=[]
def run(cand):
    rid=f'p3{cand}_{tag}_s{SEED}'
    ovr=dict(COMMON); ovr.update(CANDS[cand])
    ovr['id']=rid; ovr['output_dir']=f'{OUT}/{rid}'
    ovr['unlearn_epochs']=EPOCHS
    ovr['history_csv_path']=f'/kaggle/working/perepoch_{rid}.csv'
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
         'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
    cmd=['python','training/forgetmi_p3_cand.py','--config',CONFIG,'--seed',str(SEED),
         '--scheme','uni_nokd','--fresh','--override',arg]
    print('='*72+f'\n{rid}   {CANDS[cand]}\n'+'='*72)
    t0=time.time()
    try:
        subprocess.run(cmd,env=env,check=True)
        LOG.append((rid,'OK',round((time.time()-t0)/3600,2)))
    except subprocess.CalledProcessError as e:
        print('FAIL',rid,e.returncode); LOG.append((rid,f'FAIL{e.returncode}',round((time.time()-t0)/3600,2)))
_t=time.time()
for c in JOBS: run(c)
print(f'\nTONG {(time.time()-_t)/3600:.2f}h'); [print(' ',*x) for x in LOG]


In [ ]:
# Cell 4: eval OG + GOLD tren D_t_final (chi khi CORE; ablation khong can lai)
import os, subprocess
def evalref(label, mpath):
    ovr=dict(COMMON); ovr.pop('ce_selector',None); ovr.pop('s4_delta',None)
    ovr['output_dir']=f'{OUT}/_ref'; ovr['results_csv_path']=RESULTS
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
    cmd=['python','training/forgetmi_eval_only.py','--config',CONFIG,'--seed',str(SEED),
         '--label',label,'--model_type','pretrained','--model_path',mpath,'--method','reference','--override',arg]
    print('eval-ref',label)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL',label,e.returncode)
# Chi can chay tren MOT account (OG/GOLD giong nhau moi candidate). Dat False o 4 acc con lai.
RUN_REF = True
if RUN_REF:
    evalref(f'og_{tag}',BASE)
    if HAS_GOLD: evalref(f're_{tag}',GOLD)
    else: print('(khong co GOLD cho',tag,')')
else:
    print('RUN_REF=False -> bo qua (dung OG/GOLD tu account khac)')


In [ ]:
# Cell 5: QUY DAO d_u — tieu chi quyet dinh candidate nao dang chay tiep 30 epoch
import glob, os, pandas as pd
pd.set_option('display.width',200)
REF={'uni_nokd 30ep (cu)':[5.34,5.83,6.34,6.58]}   # E1,E5,E10,E15 tu run truoc
rows=[]
for f in sorted(glob.glob('/kaggle/working/perepoch_p3*_%s_s%d.csv'%(tag,SEED))):
    d=pd.read_csv(f); name=os.path.basename(f).replace('perepoch_','').replace('.csv','')
    g=lambda e: (d.loc[d.epoch==e,'d_u_mean'].iloc[0] if (d.epoch==e).any() else float('nan'))
    last=int(d.epoch.max())
    rows.append({'run':name,'E1':round(g(1),2),'E5':round(g(5),2),'E10':round(g(10),2),
                 f'E{last}':round(g(last),2),'tang E10->cuoi':round(g(last)-g(10),2),
                 'ihl_cuoi':round(d.ihl.iloc[-1],4),'ce_cuoi':round(d.ce.iloc[-1],3)})
if rows:
    print('===== d_u_mean theo epoch (cang tang cang tot) =====')
    print(pd.DataFrame(rows).to_string(index=False))
    print('\nMoc so sanh: uni_nokd cu bao hoa ~6.6 tai E15 -> candidate nao vuot RO va')
    print('van con tang o epoch cuoi thi moi dang chay lai 30 epoch.')
if os.path.exists(RESULTS):
    print('\n===== metric cuoi =====')
    dr=pd.read_csv(RESULTS)
    cols=[c for c in ['id','checkpoint_kind','selected_epoch','Forget_AUC','Forget_Macro_F1',
                      'Test_AUC','Test_Macro_F1','MIA','forget_ce','test_ce','trainable_ratio']
          if c in dr.columns]
    print(dr[cols].to_string(index=False))
print('\nTAI VE: perepoch_p3*.csv + results_p3imp_*.csv + OUT/*/checkpoint_selection_*/*')
